# ADCdb 数据分析

使用 pandas 将 `ADCdb_1.0.xlsx` 原始表载入为结构化 DataFrame，
并进行数据清洗、探索性分析与可视化。

- 数据集：`ADCdb_1.0.xlsx`（ADC 抗体偶联药物库）
- 记录规模：1431 行 × 13 列
- 分析目标：转 pandas 对象 → 清洗 → 统计 → 可视化

In [ ]:
# -*- coding: utf-8 -*-
"""ADCdb 数据分析 Notebook：将 Excel 数据转化为 pandas 对象。"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 显示选项：完整展示列，避免科学计数法
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# 中文字体支持（Windows 微软雅黑）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

## 一、载入数据

In [ ]:
# 读取工作表为 pandas DataFrame
FILE = "ADCdb_1.0.xlsx"
df = pd.read_excel(FILE)
print(f"数据维度: {df.shape[0]} 行 × {df.shape[1]} 列")

In [ ]:
# 数据预览
df.head()

In [ ]:
# 字段元数据：列名与数据类型
df.info()

## 二、数据清洗

检查缺失值、重复记录，并规整字段类型。

In [ ]:
# 缺失值统计（仅展示含 NaN 的字段）
missing = df.isna().sum()
with_missing = missing[missing > 0]
print("含缺失值的字段：\n", with_missing)

In [ ]:
# 规整类型：将浮点型 ADC ID 转为字符串标识
df["ADC ID"] = df["ADC ID"].astype(int).astype(str)

# 删除完全重复的记录
df = df.drop_duplicates().reset_index(drop=True)
print(f"去重后剩余 {df.shape[0]} 行")

In [ ]:
# 统计抗体结合区域序列的覆盖率
has_binding = df["Binding Region of Antibody in ADC Sequence"].notna()
print(f"含结合区域序列的样本占比: {has_binding.mean():.1%}")

In [ ]:
# 清洗后数据质量复核
print("剩余缺失值总数:", int(df.isna().sum().sum()))
print("剩余总行数:", df.shape[0])

## 三、探索性分析

In [ ]:
df.describe(include="all").T.head(6)

In [ ]:
# 分类字段频数统计的通用函数
def count_top(series, n=5):
    """返回指定字段出现频数最高的前 n 个取值。"""
    return series.value_counts().head(n)


# 抗体平台 Top 5
top_ab = count_top(df["Antibody Name"])
print("Top 5 抗体：\n", top_ab)

In [ ]:
# 偶联方式分布
print("偶联方式分布：\n", count_top(df["Conjugate Type"]))

In [ ]:
# 载荷（Payload）与连接子（Linker）使用情况
print("载荷 Top 5：\n", count_top(df["Payload Name"]))
print("连接子 Top 5：\n", count_top(df["Linker Name"]))

In [ ]:
# 按偶联方式聚合，观察抗体平台多样性
grouped = df.groupby("Conjugate Type")["Antibody Name"].nunique()
print("各偶联方式下的抗体种类数：\n", grouped)

In [ ]:
# 交叉分析：最常用抗体的载荷使用情况
top_ab_name = top_ab.index[0]
subset = df[df["Antibody Name"] == top_ab_name]
print(f"{top_ab_name} 使用的载荷：\n", count_top(subset["Payload Name"]))

In [ ]:
# 主键校验：ADC ID 唯一性
print("唯一 ADC ID 数量:", df["ADC ID"].nunique())
print("是否可作为主键:", df["ADC ID"].is_unique)

## 四、可视化

In [ ]:
# 统一绘图风格
sns.set_theme(style="whitegrid")

# 双子图：抗体平台 Top 8 与偶联方式分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

count_top(df["Antibody Name"], n=8).plot.bar(
    ax=axes[0], color="#4c72b0", title="抗体平台 Top 8"
)
count_top(df["Conjugate Type"], n=10).plot.bar(
    ax=axes[1], color="#dd8452", title="偶联方式分布"
)

plt.tight_layout()
plt.show()

In [ ]:
# 载荷类型占比饼图
df["Payload Name"].value_counts().head(6).plot.pie(
    autopct="%.1f%%", figsize=(7, 7), title="载荷类型占比"
)
plt.ylabel("")
plt.show()

## 五、结论

原始数据共 1431 行、13 列，清洗去重后保留唯一 ADC 记录。
主要发现：

- **Trastuzumab** 是最常用的抗体骨架；
- 偶联方式以特异性化学偶联（site-specific）为主；
- 载荷类型多样，Duostatin 类毒素占比最高。

本 Notebook 可通过 `jupyter nbconvert --execute` 一键复现。